# Bank of England (BoE) Financial Market Analysis

This notebook fetches, cleans, and visualizes key UK financial indicators from the official Bank of England (BoE) Interactive Statistical Database:
1. **Official Bank Rate (Base Rate)** — `IUDBEDR` (Daily)
2. **Sterling Overnight Index Average (SONIA) Rate** — `IUDSOIA` (Daily)
3. **5-Year UK Government Bond (Gilt) Nominal Par Yield** — `IUDSNPY` (Daily)
4. **10-Year UK Government Bond (Gilt) Nominal Par Yield** — `IUDMNPY` (Daily)
5. **20-Year UK Government Bond (Gilt) Nominal Par Yield** — `IUDLNPY` (Daily)

> **Note on Maturities:** In the Bank of England's Interactive Statistical Database, daily Gilt par yields are pre-calculated for three key maturity horizons: Short-term (**5-Year** — `IUDSNPY`), Medium-term (**10-Year** — `IUDMNPY`), and Long-term (**20-Year** — `IUDLNPY`). The 30-year maturity point is not published as a standalone database series code but rather as part of the BoE's monthly bulk yield curve model archives; therefore, this analysis utilizes the 5-year, 10-year, and 20-year Gilt yields to cover the entire term structure on a daily basis.

We build individual charts for each rate using **Plotly**, a dedicated **Gilt Yield Curve** chart, and finally a **Combined Financial Indicators** chart overlaying all interest rates and bond yields together.

In [1]:
import io
import requests
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Define the indicators and their Bank of England database series codes
series_config = {
    "Base Rate": {
        "code": "IUDBEDR",
        "description": "BoE Official Bank Rate (Daily)",
        "color": "#C8102E"  # Bank of England Crimson
    },
    "SONIA": {
        "code": "IUDSOIA",
        "description": "Sterling Overnight Index Average (Daily)",
        "color": "#003087"  # Deep Navy Blue
    },
    "5Y Gilt": {
        "code": "IUDSNPY",
        "description": "5-Year UK Government Bond Yield (Daily)",
        "color": "#E69F00"  # Warm Amber/Orange
    },
    "10Y Gilt": {
        "code": "IUDMNPY",
        "description": "10-Year UK Government Bond Yield (Daily)",
        "color": "#008080"  # Classic Teal
    },
    "20Y Gilt": {
        "code": "IUDLNPY",
        "description": "20-Year UK Government Bond Yield (Daily)",
        "color": "#782F40"  # Deep Plum/Maroon
    }
}

# Fetch data for the last 12 months
end_date = datetime.today()
start_date = end_date - timedelta(days=365)

data_frames = {}

print("Fetching data from the Bank of England Database...")
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
}

for name, cfg in series_config.items():
    code = cfg["code"]
    url = (
        "https://www.bankofengland.co.uk/boeapps/database/_iadb-FromShowColumns.asp"
        "?csv.x=yes"
        f"&Datefrom={start_date.strftime('%d/%b/%Y')}"
        f"&Dateto={end_date.strftime('%d/%b/%Y')}"
        f"&SeriesCodes={code}"
        "&CSVF=TN"
        "&UsingCodes=Y"
    )
    
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    
    # Read CSV
    df = pd.read_csv(io.StringIO(response.text))
    df.columns = ["Date", name]
    
    # Clean data
    df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True)
    df[name] = pd.to_numeric(df[name], errors="coerce")
    df = df.dropna().sort_values("Date").reset_index(drop=True)
    
    data_frames[name] = df
    print(f"✓ {name} ({code}) fetched successfully: {len(df)} records. Current rate: {df[name].iloc[-1]:.4f}% (as of {df['Date'].iloc[-1].strftime('%d %b %Y')})")


Fetching data from the Bank of England Database...
✓ Base Rate (IUDBEDR) fetched successfully: 251 records. Current rate: 3.7500% (as of 28 May 2026)
✓ SONIA (IUDSOIA) fetched successfully: 250 records. Current rate: 3.7290% (as of 27 May 2026)
✓ 5Y Gilt (IUDSNPY) fetched successfully: 250 records. Current rate: 4.3660% (as of 27 May 2026)
✓ 10Y Gilt (IUDMNPY) fetched successfully: 250 records. Current rate: 4.8259% (as of 27 May 2026)
✓ 20Y Gilt (IUDLNPY) fetched successfully: 250 records. Current rate: 5.3334% (as of 27 May 2026)


## Chart 1: Bank of England Official Base Rate

The Bank of England Base Rate (officially the Bank Rate) is the interest rate the BoE pays on commercial bank reserves. It is the primary tool of monetary policy in the UK. Because policy rates are set during discrete Monetary Policy Committee (MPC) meetings, they change in discrete steps.

In [2]:
df_base = data_frames.get("Base Rate")

fig_base = go.Figure()
fig_base.add_trace(go.Scatter(
    x=df_base["Date"],
    y=df_base["Base Rate"],
    mode="lines",
    line=dict(color=series_config["Base Rate"]["color"], width=3, shape="hv"),  # Step-like shape for policy changes
    name="Base Rate",
    hovertemplate="<b>%{x|%d %b %Y}</b><br>Base Rate: <b>%{y:.2f}%</b><extra></extra>"
))

fig_base.update_layout(
    title=dict(
        text="<b>Bank of England Official Bank Rate (Base Rate)</b><br><sup>Daily policy rate over the last 12 months</sup>",
        font=dict(size=18, family="Arial"),
        x=0.5,
        y=0.92
    ),
    xaxis=dict(
        title="Date",
        showgrid=True,
        gridcolor="#F0F0F0",
        linecolor="#D3D3D3",
        tickformat="%b %Y"
    ),
    yaxis=dict(
        title="Interest Rate (%)",
        showgrid=True,
        gridcolor="#F0F0F0",
        linecolor="#D3D3D3",
        ticksuffix="%",
        zeroline=False
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    margin=dict(t=80, b=80, l=60, r=40),
    annotations=[dict(
        text='Source: <a href="https://www.bankofengland.co.uk/monetary-policy/the-bank-rate">Bank of England (Series IUDBEDR)</a>',
        showarrow=False, xref="paper", yref="paper",
        x=1, y=-0.15, xanchor="right", font=dict(size=10, color="grey")
    )]
)

fig_base.show()


## Chart 2: SONIA (Sterling Overnight Index Average)

SONIA is the effective overnight interest rate paid by banks for unsecured sterling transactions. It is a critical benchmark rate reflecting actual funding conditions in the short-term wholesale money markets.

In [3]:
df_sonia = data_frames.get("SONIA")

fig_sonia = go.Figure()
fig_sonia.add_trace(go.Scatter(
    x=df_sonia["Date"],
    y=df_sonia["SONIA"],
    mode="lines",
    line=dict(color=series_config["SONIA"]["color"], width=2),
    name="SONIA",
    hovertemplate="<b>%{x|%d %b %Y}</b><br>SONIA Rate: <b>%{y:.4f}%</b><extra></extra>"
))

fig_sonia.update_layout(
    title=dict(
        text="<b>Sterling Overnight Index Average (SONIA)</b><br><sup>Daily unsecured overnight interest rate over the last 12 months</sup>",
        font=dict(size=18, family="Arial"),
        x=0.5,
        y=0.92
    ),
    xaxis=dict(
        title="Date",
        showgrid=True,
        gridcolor="#F0F0F0",
        linecolor="#D3D3D3",
        tickformat="%b %Y"
    ),
    yaxis=dict(
        title="SONIA Rate (%)",
        showgrid=True,
        gridcolor="#F0F0F0",
        linecolor="#D3D3D3",
        ticksuffix="%",
        zeroline=False
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    margin=dict(t=80, b=80, l=60, r=40),
    annotations=[dict(
        text='Source: <a href="https://www.bankofengland.co.uk/markets/sonia-benchmark">Bank of England (Series IUDSOIA)</a>',
        showarrow=False, xref="paper", yref="paper",
        x=1, y=-0.15, xanchor="right", font=dict(size=10, color="grey")
    )]
)

fig_sonia.show()


## Chart 3: UK Government Bond (Gilt) Yields

This chart displays **only** UK Gilt yields across different maturities (5-Year, 10-Year, and 20-Year). By plotting these maturities together on one timeline, we can visualize the shape and evolution of the **UK Gilt Yield Curve** over the past year. 

A narrowing spread between long-term and short-term yields indicates a flattening curve, which often occurs as central banks raise interest rates, while an inverted curve (short-term yields higher than long-term yields) historically suggests expectations of future rate cuts.

In [4]:
df_5y = data_frames.get("5Y Gilt")
df_10y = data_frames.get("10Y Gilt")
df_20y = data_frames.get("20Y Gilt")

# Merge Gilt dataframes on Date
df_gilts = pd.merge(df_5y, df_10y, on="Date", how="outer")
df_gilts = pd.merge(df_gilts, df_20y, on="Date", how="outer")
df_gilts = df_gilts.sort_values("Date").reset_index(drop=True)

fig_gilts = go.Figure()

# 5Y Gilt
fig_gilts.add_trace(go.Scatter(
    x=df_gilts["Date"],
    y=df_gilts["5Y Gilt"],
    mode="lines",
    line=dict(color=series_config["5Y Gilt"]["color"], width=2),
    name="5Y Gilt Yield (Short-term)",
    hovertemplate="5Y Gilt Yield: <b>%{y:.4f}%</b>"
))

# 10Y Gilt
fig_gilts.add_trace(go.Scatter(
    x=df_gilts["Date"],
    y=df_gilts["10Y Gilt"],
    mode="lines",
    line=dict(color=series_config["10Y Gilt"]["color"], width=2),
    name="10Y Gilt Yield (Medium-term)",
    hovertemplate="10Y Gilt Yield: <b>%{y:.4f}%</b>"
))

# 20Y Gilt
fig_gilts.add_trace(go.Scatter(
    x=df_gilts["Date"],
    y=df_gilts["20Y Gilt"],
    mode="lines",
    line=dict(color=series_config["20Y Gilt"]["color"], width=2),
    name="20Y Gilt Yield (Long-term)",
    hovertemplate="20Y Gilt Yield: <b>%{y:.4f}%</b>"
))

fig_gilts.update_layout(
    title=dict(
        text="<b>UK Government Bond (Gilt) Yield Curve Evolution</b><br><sup>Daily 5-Year, 10-Year, and 20-Year nominal par yields over the last 12 months</sup>",
        font=dict(size=18, family="Arial"),
        x=0.5,
        y=0.92
    ),
    xaxis=dict(
        title="Date",
        showgrid=True,
        gridcolor="#F0F0F0",
        linecolor="#D3D3D3",
        tickformat="%b %Y"
    ),
    yaxis=dict(
        title="Yield (%)",
        showgrid=True,
        gridcolor="#F0F0F0",
        linecolor="#D3D3D3",
        ticksuffix="%",
        zeroline=False
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#E5E5E5",
        borderwidth=1
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    margin=dict(t=100, b=80, l=60, r=40),
    annotations=[dict(
        text='Source: <a href="https://www.bankofengland.co.uk/statistics">Bank of England Database</a> (Series: IUDSNPY, IUDMNPY, IUDLNPY)',
        showarrow=False, xref="paper", yref="paper",
        x=1, y=-0.15, xanchor="right", font=dict(size=10, color="grey")
    )]
)

fig_gilts.show()


## Chart 4: Combined Financial Indicators Chart

This unified overlay chart combines **all five series** onto a single timeline. 

By overlaying monetary policy (**Base Rate**), short-term risk-free funding (**SONIA**), and UK sovereign borrowing yields (**5Y, 10Y, and 20Y Gilts**), we can inspect key market relationships on any given business day:
1. **Monetary Transmission Efficiency**: The tight spread between the official **Base Rate** steps and **SONIA**'s daily movements.
2. **Term Premium & Risk Premium**: The spread between immediate overnight rates (**SONIA**) and sovereign Gilt yields across the short-term (5Y), medium-term (10Y), and long-term (20Y) horizons.

In [5]:
# Merge all five dataframes on Date to create a unified timeline
df_combined = pd.merge(df_base, df_sonia, on="Date", how="outer")
df_combined = pd.merge(df_combined, df_5y, on="Date", how="outer")
df_combined = pd.merge(df_combined, df_10y, on="Date", how="outer")
df_combined = pd.merge(df_combined, df_20y, on="Date", how="outer")
df_combined = df_combined.sort_values("Date").reset_index(drop=True)

# Forward-fill the Base Rate so it maintains its step appearance
df_combined["Base Rate"] = df_combined["Base Rate"].ffill()

fig_combined = go.Figure()

# 1. Base Rate (thick discrete steps)
fig_combined.add_trace(go.Scatter(
    x=df_combined["Date"],
    y=df_combined["Base Rate"],
    mode="lines",
    line=dict(color=series_config["Base Rate"]["color"], width=3.5, shape="hv"),
    name="Base Rate (Policy)",
    hovertemplate="Base Rate: <b>%{y:.2f}%</b>"
))

# 2. SONIA (thin overnight daily)
fig_combined.add_trace(go.Scatter(
    x=df_combined["Date"],
    y=df_combined["SONIA"],
    mode="lines",
    line=dict(color=series_config["SONIA"]["color"], width=1.5),
    name="SONIA (Overnight)",
    hovertemplate="SONIA: <b>%{y:.4f}%</b>"
))

# 3. 5Y Gilt
fig_combined.add_trace(go.Scatter(
    x=df_combined["Date"],
    y=df_combined["5Y Gilt"],
    mode="lines",
    line=dict(color=series_config["5Y Gilt"]["color"], width=2),
    name="5Y Gilt Yield",
    hovertemplate="5Y Gilt Yield: <b>%{y:.4f}%</b>"
))

# 4. 10Y Gilt
fig_combined.add_trace(go.Scatter(
    x=df_combined["Date"],
    y=df_combined["10Y Gilt"],
    mode="lines",
    line=dict(color=series_config["10Y Gilt"]["color"], width=2),
    name="10Y Gilt Yield",
    hovertemplate="10Y Gilt Yield: <b>%{y:.4f}%</b>"
))

# 5. 20Y Gilt
fig_combined.add_trace(go.Scatter(
    x=df_combined["Date"],
    y=df_combined["20Y Gilt"],
    mode="lines",
    line=dict(color=series_config["20Y Gilt"]["color"], width=2),
    name="20Y Gilt Yield",
    hovertemplate="20Y Gilt Yield: <b>%{y:.4f}%</b>"
))

fig_combined.update_layout(
    title=dict(
        text="<b>BoE Base Rate, SONIA & UK Gilt Yield Curve Comparison</b><br><sup>Comprehensive overlay of monetary policy, wholesale money markets, and sovereign borrowing costs</sup>",
        font=dict(size=20, family="Arial"),
        x=0.5,
        y=0.92
    ),
    xaxis=dict(
        title="Date",
        showgrid=True,
        gridcolor="#F0F0F0",
        linecolor="#D3D3D3",
        tickformat="%b %Y"
    ),
    yaxis=dict(
        title="Yield / Interest Rate (%)",
        showgrid=True,
        gridcolor="#F0F0F0",
        linecolor="#D3D3D3",
        ticksuffix="%",
        zeroline=False
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#E5E5E5",
        borderwidth=1
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    margin=dict(t=100, b=80, l=60, r=40),
    annotations=[dict(
        text='Source: <a href="https://www.bankofengland.co.uk/statistics">Bank of England Database</a> (IUDBEDR, IUDSOIA, IUDSNPY, IUDMNPY, IUDLNPY)',
        showarrow=False, xref="paper", yref="paper",
        x=1, y=-0.15, xanchor="right", font=dict(size=10, color="grey")
    )]
)

fig_combined.show()
